# DAG Parameterization
The goal of this notebook is to assess which of the graphs (DAGs or PAGs) we generated during our causal structure learning step produces the best predictive model.

In [ ]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import glob

# Bayesian Networks
from pgmpy.models import LinearGaussianBayesianNetwork

# Metrics
from sklearn.calibration import calibration_curve
from pycalib.metrics import ECE

import xgboost
import shap

## Data preparation

In [ ]:
# Load training data
X_train = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_imp.csv', index_col=0)
X_test = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_imp.csv', index_col=0)

print(X_train.shape, X_test.shape)

#Remove features with 0 variance
X_train = X_train.loc[:, X_train.var() >= 0.01]
X_test = X_test.filter(items=X_train.columns) # Keep only columns in X_train
print(X_train.shape, X_test.shape)

# # Impute missing values using Iterative Imputer (Linear Gaussian BN can't handle missing values)
# imputer = IterativeImputer(random_state=42, max_iter=100)
# X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
# X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_imp = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_mice.csv', index_col=0)
X_test_imp = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_mice.csv', index_col=0)

y_train = pd.read_csv('../../Data Pre-processing/y_train_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()
y_test = pd.read_csv('../../Data Pre-processing/y_test_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()

# Correlation feature selection
correlation_threshold = 0.3
X_train_corr = X_train_imp.loc[:, X_train_imp.corrwith(y_train['Outcome']).abs() >= correlation_threshold]
print(X_train_corr.shape)

# Loading DAGs

In [ ]:
#Get all adjacency files
adjacency_files = glob.glob("../DAGs/*_adjacency.csv")

dags = {}
dags['Control'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_imp.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
dags['Correlation'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_corr.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
for file in adjacency_files:
    dag_name = re.search(r'../DAGs/(.*)_adjacency.csv', file).group(1)
    dag_name = re.sub(r'(?<![a-zA-Z])x(?![a-zA-Z])', lambda _: '$\\cap$', dag_name)
    dag_name = dag_name.replace(' + ', ' $\\cup$ ')
    dags[dag_name] = nx.DiGraph(pd.read_csv(file, index_col=0))

dags.pop('Golem $\\cap$ PCMB')  # Remove problematic DAG (has 0 nodes associated with Outcome)
list(dags.keys())

In [ ]:
list(dags['Simplified Golem'].nodes())

# Model Training Functions
For each dag we train a Linear Gaussian Bayesian Network, an XGBoost

In [ ]:
from MLstatkit import Bootstrapping, Delong_test, Permutation_test
from itertools import combinations

def compare_models(y, y_prob1, y_prob2):
        # DeLong Test for AUROC
        z, p = Delong_test(y.values, y_prob1, y_prob2, return_ci=False, return_auc=False)
        metric_a, metric_b, p_value, benchmark, samples_mean, samples_std = Permutation_test(y.values, y_prob1, y_prob2, metric_str='average_precision')
        return (z, p), (metric_a, metric_b, p_value, benchmark, samples_mean, samples_std)

def performance_report(X, y, model):
    try:
        y_prob = model.predict_proba(X)[:, 1] # Predict on test data
    except AttributeError:
        y_prob = model.predict(X)['Outcome'].to_numpy() # Predict on test data (LGBN fallback: pgmpy 1.x returns DataFrame w/ Outcome conditional mean)
        y_prob = np.where(y_prob > 1, 1, np.where(y_prob < 0, 0, y_prob))

    ## Calculate Performance Metrics with Bootstrapping
    n_bootstraps = 1000
    # Average Precision
    ap, ap_lower, ap_upper = Bootstrapping(y.values, y_prob, metric_str='average_precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # AUROC
    auroc, auroc_lower, auroc_upper = Bootstrapping(y.values, y_prob, metric_str='roc_auc', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # Precision 
    precision, precision_lower, precision_upper = Bootstrapping(y.values, y_prob, metric_str='precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # Recall
    recall, recall_lower, recall_upper = Bootstrapping(y.values, y_prob, metric_str='recall', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # F1 Score
    f1, f1_lower, f1_upper = Bootstrapping(y.values, y_prob, metric_str='f1', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # Accuracy
    accuracy, accuracy_lower, accuracy_upper = Bootstrapping(y.values, y_prob, metric_str='accuracy', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    # ECE
    ece = ECE(y.values, y_prob.reshape(-1, 1), bins=50)

    return {'Average Precision': f"{ap:.2f} ({ap_lower:.2f}, {ap_upper:.2f})",
     'AUROC': f"{auroc:.2f} ({auroc_lower:.2f}, {auroc_upper:.2f})",
     'Precision': f"{precision:.2f} ({precision_lower:.2f}, {precision_upper:.2f})",
     'Recall': f"{recall:.2f} ({recall_lower:.2f}, {recall_upper:.2f})",
     'F1 Score': f"{f1:.2f} ({f1_lower:.2f}, {f1_upper:.2f})",
     'Accuracy': f"{accuracy:.2f} ({accuracy_lower:.2f}, {accuracy_upper:.2f})",
     'ECE': f"{ece:.3f}"}

def train_models(remove_drugs=False, remove_interventions=False):
        results = []
        shap_values = {}
        calibrations = {}
        model_preds = {}
        drugs = ['Midazolam', 'Fentanyl', 'Olanzapine', 'Haloperidol', 
                'Dexmedetomidine', 'Lorazepam', 'Morphine', 'Hydromorphone', 
                'Dopamine', 'Cisatracurium', 'Epinephrine', 'Norepinephrine', 
                'Milrinone', 'Dobutamine']
        

        for dag_name, dag in dags.items():
        
                # Simplified is not in the dag_name, skip
                if 'Simplified' not in dag_name and dag_name not in ['Control', 'Correlation']:
                        continue


                if dag is not None:
                        print(f"Processing {dag_name} with {dag.number_of_nodes()} nodes and {dag.number_of_edges()} edges")

                nodes_in_dag = list(dag.nodes())

                if 'Simplified' in dag_name or dag_name in ['Control', 'Correlation']:
                        if remove_drugs:
                                for drug in drugs:
                                        if drug in nodes_in_dag:
                                                nodes_in_dag.remove(drug)
                
                        if remove_interventions:
                                intervention_nodes = ['CRRT Therapy Type', 'ECMO Type', 'Ventilated', 'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube']
                                for intervention in intervention_nodes:
                                        if intervention in nodes_in_dag:
                                                nodes_in_dag.remove(intervention)

                        if dag_name == 'Correlation':
                                features_in_dag = [col for col in X_train_corr.columns if any(re.search(rf'^{re.escape(node)}(_.+)?$', col) for node in nodes_in_dag)]
                        else:
                                features_in_dag = [col for col in X_train_imp.columns if any(re.search(rf'^{re.escape(node)}(_.+)?$', col) for node in nodes_in_dag)]

                        train_dag = nx.DiGraph()

                        # Map DAG nodes to features
                        for node in nodes_in_dag:
                                ends = list(dag.successors(node))
                                from_features = [col for col in features_in_dag + ['Outcome'] if re.search(rf'^{re.escape(node)}(_.+)?$', col)]
                                to_features = [col for col in features_in_dag + ['Outcome'] if any(re.search(rf'^{re.escape(end)}(_.+)?$', col) for end in ends)]
                                for from_feature in from_features:
                                        for to_feature in to_features:
                                                train_dag.add_edge(from_feature, to_feature)
                        
                        n_biomarkers = len(nodes_in_dag) - 1  # Exclude Outcome
                else:
                        # Use list comprehensions to avoid mutating nodes_in_dag while iterating
                        if remove_drugs:
                                nodes_in_dag = [node for node in nodes_in_dag if not any(drug in node for drug in drugs)]
                        if remove_interventions:
                                intervention_nodes = ['CRRT Therapy Type', 'ECMO Type', 'Ventilated', 'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube']
                                nodes_in_dag = [node for node in nodes_in_dag if not any(intervention in node for intervention in intervention_nodes)]
                        
                        features_in_dag = X_train_imp.filter(items=nodes_in_dag).columns.tolist()
                        train_dag = dag.subgraph(features_in_dag + ['Outcome']).copy()
                        n_biomarkers = X_train_imp.filter(items=nodes_in_dag).columns.str.replace(r'(_.+)?$', '', regex=True).nunique() - 1  # Exclude Outcome

                # XGB trains on non-imputed X_train; restrict to columns actually present there
                xgb_features = [f for f in features_in_dag if f in X_train.columns]

                models = {}
                # Instantiate Models
                models['LGBN'] = LinearGaussianBayesianNetwork(train_dag.edges())
                # models['LR'] = LogisticRegression(max_iter=1000, random_state=42)
                models['XGB'] = xgboost.XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='aucpr')
                

                for model_name, model in models.items():
                        metrics = {}
                        metrics['Model'] = model_name
                        metrics['DAG'] = dag_name

                        if model_name == 'LGBN':
                                # Fit the model
                                train = pd.concat([X_train_imp.filter(features_in_dag), y_train.filter(['Outcome']).astype(int)], axis=1)
                                model.fit(train)
                                metrics.update(performance_report(X_test_imp.filter(features_in_dag), y_test.Outcome, model))
                                metrics['# Features'] = len(features_in_dag)
                        elif model_name == 'LR':
                                model.fit(X_train_imp.filter(features_in_dag), y_train.Outcome)
                                model_preds[dag_name] = model.predict_proba(X_test_imp.filter(features_in_dag))[:, 1]
                                metrics.update(performance_report(X_test_imp.filter(features_in_dag), y_test.Outcome, model))
                                metrics['# Features'] = len(features_in_dag)
                        else:
                                X_train_filtered = X_train.filter(xgb_features)
                                X_test_filtered = X_test.filter(xgb_features)
                                model.fit(X_train_filtered, y_train.Outcome)
                                y_prob = model.predict_proba(X_test_filtered)[:, 1]
                                model_preds[dag_name] = y_prob
                                metrics.update(performance_report(X_test_filtered, y_test.Outcome, model))
                                calibrations[dag_name] = calibration_curve(y_test.Outcome, y_prob, n_bins=50)
                                metrics['# Features'] = len(xgb_features)

                        metrics['# Biomarkers'] = n_biomarkers

                        results.append(metrics)


                # SHAP feature importance
                X_test_shap = X_test.filter(xgb_features)
                explainer = shap.TreeExplainer(models['XGB'])
                values = explainer.shap_values(X_test_shap)
                values = pd.DataFrame(values, columns=X_test_shap.columns, index=X_test_shap.index)
                shap_values[dag_name] = values
        

        return results, shap_values, calibrations, model_preds

In [ ]:
def shap_values_biomarker_summary(shap_values):
    vals = {} 
    for dag_name in shap_values:
        df = shap_values[dag_name].reset_index().melt(id_vars='uid')
        df.variable = df.variable.str.replace(r'(_.+)?$', '', regex=True)
        df.value = df.value.abs()                                   # abs before summing to prevent sign cancellation
        df = df.groupby(['uid', 'variable']).sum().reset_index()
        df = df.groupby('variable').value.mean().sort_values(ascending=True)
        vals[dag_name] = df
    return vals

# Model Training

## Bootstrapping
We simulate the potential performance of our models on an out of distribution dataset by performing a weighted bootstrap of our Test set. <br>
For context, our test set is already based on prospective samples. The training set comes from patients admitted up to 2019 and the test set is from patient admitted on or after 2020.

## Expriment 1: Comparing Feature Selection Approaches (DAGs)

In [ ]:
results, shap_values, calibrations, model_preds = train_models(remove_drugs=False, remove_interventions=False)

In [ ]:
aurocs = []
auprcs = []
print(len(model_preds))
# Model Comparisons using DeLong Test
for (model1_name, model2_name) in combinations(model_preds.keys(), 2):
        if model1_name != 'Control': continue
        model1 = model_preds[model1_name]
        model2 = model_preds[model2_name]
        comparison_result1, comparison_result2 = compare_models(y_test.Outcome, model1, model2)
        aurocs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Z-score': f"{comparison_result1[0]:.4f}", 'P-value': f"{comparison_result1[1]:.3e}", 'p_numeric': comparison_result1[1]})
        auprcs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Avg Precision Model 1': f"{comparison_result2[0]:.4f}", 'Avg Precision Model 2': f"{comparison_result2[1]:.4f}", 'P-value': f"{comparison_result2[2]:.3e}", 'Benchmark': comparison_result2[3], 'Samples Mean': f"{comparison_result2[4]:.4f}", 'Samples Std': f"{comparison_result2[5]:.4f}"})
aurocs_df = pd.DataFrame(aurocs)
auprcs_df = pd.DataFrame(auprcs)
aurocs_df.to_csv('biomarker_counts_fixed/delong_test_biomarker_selection_auroc.csv', index=False)
auprcs_df.to_csv('biomarker_counts_fixed/delong_test_biomarker_selection_auprc.csv', index=False)
aurocs_df

In [ ]:
pd.DataFrame(results).to_csv('biomarker_counts_fixed/Biomarker Selection.csv', index=False)
pd.DataFrame(results)

In [ ]:
# Plot calibration curves
for dag_name in calibrations:
    plt.figure(dpi=300, layout='constrained', figsize=(8, 5))
    plt.rcParams['font.size'] = 18
    prob_true, prob_pred = calibrations[dag_name]
    plt.plot(prob_pred, prob_true, marker='o', label='Calibration Curve')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')
    plt.xticks(np.arange(0, 1.1, 0.1))
    plt.grid()
    plt.savefig(f'biomarker_counts_fixed/Calibration Curves/Biomarker Selection - {dag_name.replace(" ", "_").replace("$\\cap$", "AND").replace("$\\cup$", "OR")}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
vals = shap_values_biomarker_summary(shap_values)
auprc_top10 = pd.DataFrame(results).sort_values(by=['Average Precision'], ascending=False).head(10).DAG.unique().tolist()
plt.rcParams.update({  
'figure.figsize': (8, 10),   # Default figure size (inches)  
'figure.dpi': 300,          # Default DPI for figure creation  
'ytick.labelsize': 16,
'xtick.labelsize': 16,
'font.size': 18,
'lines.linewidth': 2,       # Default line width  
'figure.constrained_layout.use': True,
'pdf.fonttype': 42
}) 
for dag_name in auprc_top10:
    df = vals[dag_name].sort_values(ascending=True)
    plt.barh(df.index, df)
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Biomarker')
    plt.savefig(f'biomarker_counts_fixed/Feature Importance/Biomarker Selection - {dag_name}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

## Experiment 2: No Drugs

In [ ]:
results, shap_values, calibrations, model_preds = train_models(remove_drugs=True, remove_interventions=False)

In [ ]:
pd.DataFrame(results).to_csv('biomarker_counts_fixed/No Drugs.csv', index=False)
pd.DataFrame(results)

In [ ]:
aurocs = []
auprcs = []
print(len(model_preds))
# Model Comparisons using DeLong Test
for (model1_name, model2_name) in combinations(model_preds.keys(), 2):
        model1 = model_preds[model1_name]
        model2 = model_preds[model2_name]
        comparison_result1, comparison_result2 = compare_models(y_test.Outcome, model1, model2)
        aurocs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Z-score': f"{comparison_result1[0]:.4f}", 'P-value': f"{comparison_result1[1]:.3e}", 'p_numeric': comparison_result1[1]})
        auprcs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Avg Precision Model 1': f"{comparison_result2[0]:.4f}", 'Avg Precision Model 2': f"{comparison_result2[1]:.4f}", 'P-value': f"{comparison_result2[2]:.3e}", 'Benchmark': comparison_result2[3], 'Samples Mean': f"{comparison_result2[4]:.4f}", 'Samples Std': f"{comparison_result2[5]:.4f}"})
aurocs_df = pd.DataFrame(aurocs)
auprcs_df = pd.DataFrame(auprcs)
aurocs_df.to_csv('biomarker_counts_fixed/delong_test_no_drugs_auroc.csv', index=False)
auprcs_df.to_csv('biomarker_counts_fixed/delong_test_no_drugs_auprc.csv', index=False)

In [ ]:
# Plot calibration curves
for dag_name in calibrations:
    plt.figure(dpi=300, layout='constrained', figsize=(8, 5))
    plt.rcParams['font.size'] = 18
    prob_true, prob_pred = calibrations[dag_name]
    plt.plot(prob_pred, prob_true, marker='o', label=f'Calibration Curve')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')
    # plt.title(f'No Drugs - {dag_name}')
    plt.xticks(np.arange(0, 1.1, 0.1))
    plt.grid()
    plt.savefig(f'biomarker_counts_fixed/Calibration Curves/No Drugs - {dag_name.replace(" ", "_").replace("$\\cap$", "AND").replace("$\\cup$", "OR")}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
vals = shap_values_biomarker_summary(shap_values)
auprc_top10 = pd.DataFrame(results).sort_values(by=['Average Precision'], ascending=False).head(10).DAG.unique().tolist()

plt.rcParams.update({  
'figure.figsize': (8, 10),   # Default figure size (inches)  
'figure.dpi': 300,          # Default DPI for figure creation  
'ytick.labelsize': 16,
'xtick.labelsize': 16,
'font.size': 18,
'lines.linewidth': 2,       # Default line width  
'figure.constrained_layout.use': True,
'pdf.fonttype': 42
}) 
for dag_name in auprc_top10:
    df = vals[dag_name].sort_values(ascending=True)
    plt.barh(df.index, df)
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Biomarker')
    plt.savefig(f'biomarker_counts_fixed/Feature Importance/No Drugs - {dag_name}.pdf', bbox_inches='tight', dpi=300, transparent=False)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

## Experiment 3: No Drugs or Interventions

In [ ]:
results, shap_values, calibrations, model_preds = train_models(remove_drugs=True, remove_interventions=True)

In [ ]:
pd.DataFrame(results).to_csv('biomarker_counts_fixed/No Drugs or Interventions.csv', index=False)
pd.DataFrame(results)

In [ ]:
aurocs = []
auprcs = []
print(len(model_preds))
# Model Comparisons using DeLong Test
for (model1_name, model2_name) in combinations(model_preds.keys(), 2):
        model1 = model_preds[model1_name]
        model2 = model_preds[model2_name]
        comparison_result1, comparison_result2 = compare_models(y_test.Outcome, model1, model2)
        aurocs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Z-score': f"{comparison_result1[0]:.4f}", 'P-value': f"{comparison_result1[1]:.3e}", 'p_numeric': comparison_result1[1]})
        auprcs.append({'Model 1': model1_name, 'Model 2': model2_name, 'Avg Precision Model 1': f"{comparison_result2[0]:.4f}", 'Avg Precision Model 2': f"{comparison_result2[1]:.4f}", 'P-value': f"{comparison_result2[2]:.3e}", 'Benchmark': comparison_result2[3], 'Samples Mean': f"{comparison_result2[4]:.4f}", 'Samples Std': f"{comparison_result2[5]:.4f}"})
aurocs_df = pd.DataFrame(aurocs)
auprcs_df = pd.DataFrame(auprcs)
aurocs_df.to_csv('biomarker_counts_fixed/delong_test_only_vitals_labs_auroc.csv', index=False)
auprcs_df.to_csv('biomarker_counts_fixed/delong_test_only_vitals_labs_auprc.csv', index=False)

In [ ]:
# Plot calibration curves
for dag_name in calibrations:
    plt.figure(dpi=300, layout='constrained', figsize=(8, 5))
    plt.rcParams['font.size'] = 18
    prob_true, prob_pred = calibrations[dag_name]
    plt.plot(prob_pred, prob_true, marker='o', label=f'Calibration Curve')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')
    # plt.title(f'Only vitals and labs - {dag_name}')
    # plt.legend()
    plt.xticks(np.arange(0, 1.1, 0.1))
    plt.grid()
    plt.savefig(f'biomarker_counts_fixed/Calibration Curves/Only vitals and labs - {dag_name.replace(" ", "_").replace("$\\cap$", "AND").replace("$\\cup$", "OR")}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
vals = shap_values_biomarker_summary(shap_values)
auprc_top10 = pd.DataFrame(results).sort_values(by=['Average Precision'], ascending=False).head(10).DAG.unique().tolist()
plt.rcParams.update({  
'figure.figsize': (8, 10),   # Default figure size (inches)  
'figure.dpi': 300,          # Default DPI for figure creation  
'ytick.labelsize': 16,
'xtick.labelsize': 16,
'font.size': 18,
'lines.linewidth': 2,       # Default line width  
'figure.constrained_layout.use': True,
'pdf.fonttype': 42
}) 
for dag_name in auprc_top10:
    df = vals[dag_name].sort_values(ascending=True)
    plt.barh(df.index, df)
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Biomarker')
    plt.savefig(f'biomarker_counts_fixed/Feature Importance/Only vitals and labs - {dag_name}.pdf', bbox_inches='tight', dpi=300)
    plt.close()
plt.rcParams.update(plt.rcParamsDefault)